In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import numpy as np
from steps.master_tdt_analysis import TdtExperiment

import pyqtgraph as pg

In [3]:
# =============================================================================
# EXPERIMENT SETTINGS
# =============================================================================

EXPERIMENT_NAME = "20191216"
SAMPLE_DELAY = 22

EXPERIMENT_ROOT = (
        Path(r"D:\ImThera\data_raw")
        / EXPERIMENT_NAME
)

EXPERIMENT_LOG = (
        EXPERIMENT_ROOT
        / "Notes"
        / f"{EXPERIMENT_NAME}_Experimental_Log.xlsx"
)

OUTPUT_DIR = (
        Path(r"D:\ImThera\data_processed")
        / EXPERIMENT_NAME
)

STORES = [
    "RawE",
    "RawG",
]

RECORDING_CHANNEL_NAMES = [
    "LIFE 1",
    "LIFE 2",
    "LIFE 3",
    "LIFE 4",
    "EMG 1",
    "EMG 2",
    "EMG 3",
]

RECORDING_CHANNEL_TYPES = [
    "ENG",
    "ENG",
    "ENG",
    "ENG",
    "EMG",
    "EMG",
    "EMG",
]

REMOVE_CHANNELS: list[str | int] = ["RawG 4"]

FILTER_MEDIAN_LOWPASS = False
FILTER_MEDIAN_HIGHPASS = True
FILTER_GAUSSIAN_HIGHPASS = True
FILTER_POWERLINE = True
FILTER_POST_AVERAGE = False
TDT_CHUNK_SIZE = 2_000_000


In [4]:
# =============================================================================
# LOAD EXPERIMENT
# =============================================================================

def load_experiment() -> Any:
    """
    Load, curate, and prepare the experiment.
    """
    if not EXPERIMENT_ROOT.exists():
        raise FileNotFoundError(
            f"Experiment folder not found: {EXPERIMENT_ROOT}"
        )

    if not EXPERIMENT_LOG.exists():
        raise FileNotFoundError(
            f"Experimental log not found: {EXPERIMENT_LOG}"
        )

    experiment = TdtExperiment(
        experiment_name=EXPERIMENT_NAME,
        experiment_storage_path=str(
            EXPERIMENT_ROOT
        ),
        tdt_chunk_size=TDT_CHUNK_SIZE,
        stores=STORES,
        sample_delay=SAMPLE_DELAY,
    )

    experiment.curate_data(
        ch_names=RECORDING_CHANNEL_NAMES,
        ch_types=RECORDING_CHANNEL_TYPES,
        remove_channels=REMOVE_CHANNELS,
        filter_median_low=FILTER_MEDIAN_LOWPASS,
        filter_median=FILTER_MEDIAN_HIGHPASS,
        filter_gaussian=FILTER_GAUSSIAN_HIGHPASS,
        filter_powerline=FILTER_POWERLINE,
    )

    experiment.gather_ecap(
        experiment_log_path=str(
            EXPERIMENT_LOG
        ),
        filter_post_average=FILTER_POST_AVERAGE,
        plot_AUCs=False,
    )

    return experiment

In [5]:
import time
start_time = time.perf_counter()
experiment = load_experiment()
print(f"Execution time: {time.perf_counter() - start_time}")


D:\PycharmProjects\pyeCAP\pyeCAP\ephys.py:152: UserWarning: Channel lengths differ by one TDT block; trimming to the shortest channel.
  data_store = TdtArray(


Execution time: 10.780708099991898


In [6]:
result = experiment.data_ecap.epoch(
    condition="Intact",
    pulse_amplitude="max",
)

In [7]:
t1 = np.mean(result.array, axis=1)
t1.shape

(4, 7, 977)

In [8]:
# print(f"Graph tasks for source: {len(experiment.data_ephys.array.dask):,}")
# print(f"Graph tasks for individual epoch array: {len(experiment.data_ecap.dask_array((0,0))):,}")

In [9]:
# start_time = time.perf_counter()
# mean_traces = experiment.data_ecap.compute_mean_traces()
# print(f"EMean traces computed in: {time.perf_counter() - start_time}")

In [10]:
# start = time.perf_counter()
# mean_traces_again = experiment.data_ecap.compute_mean_traces()
# print(f"Cached call: {time.perf_counter() - start:.6f} seconds")
#
# print(mean_traces_again is experiment.data_ecap.mean_traces)

In [11]:
# result = experiment.data_ecap.epoch(
#     condition="Intact",
#     pulse_amplitude="max",
#     recording_channels=["LIFE " + str(i) for i in range(1,5)],
# )

In [12]:
# result = experiment.data_ecap.epoch(
#     condition="Intact",
#     pulse_amplitude="max",
#     channel="Channel 1",
# #    recording_channels="LIFE 1",
# )

In [13]:
experiment.data_ecap.plot_recording_channels_interactive(
    amplitude=1500,
    condition="Intact",
    relative_time_frame=(-0.0001,0.004),
    stimulation_contact="Channel 1",
    plot_window="aalpha",
)

C:\Users\steph\AppData\Local\Temp\ipykernel_24380\135289579.py:1: UserWarning: Requested amplitude 1500 µA does not exist. Using nearest available amplitude: 1000 µA.
  experiment.data_ecap.plot_recording_channels_interactive(


(<pyqtgraph.widgets.GraphicsLayoutWidget.GraphicsLayoutWidget at 0x23195882710>,
 (np.int64(0), np.int64(1)))